In [ ]:
!nvidia-smi

In [ ]:
import torch
print("GPU:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
!pip install diffusers transformers accelerate TTS audiocraft moviepy gTTS sentencepiece huggingface_hub

In [ ]:
!ffmpeg -version

In [ ]:
import os

BASE="/kaggle/working/ai_video_factory"

folders=[
"scenes",
"clips",
"audio",
"music",
"video"
]

for f in folders:
    os.makedirs(f"{BASE}/{f}",exist_ok=True)

BASE

In [ ]:
import os

INPUT_PATH = "/kaggle/input/transcript/transcript.txt"

with open(INPUT_PATH,"r") as f:
    transcript = f.read()

print("Transcript loaded. Length:", len(transcript))

In [ ]:
def split_scenes(text,words=8):

    w=text.split()
    scenes=[]

    for i in range(0,len(w),words):
        scenes.append(" ".join(w[i:i+words]))

    return scenes

scenes=split_scenes(transcript)

print("Total scenes:",len(scenes))
scenes[:5]

In [ ]:
import random

# ---------------------------------------------------
# Visual curiosity spikes (high retention triggers)
# ---------------------------------------------------

visual_spikes = [

"gigantic artificial intelligence supercomputer glowing inside a dark futuristic datacenter with endless server racks",

"extreme close up of humanoid robot eyes reflecting neural networks and digital code",

"earth from space covered in glowing artificial intelligence data networks connecting continents",

"massive underground server facility powering the world's artificial intelligence systems",

"futuristic megacity skyline controlled by artificial intelligence infrastructure",

"scientists observing a giant holographic AI brain simulation inside a dark research lab",

"glowing digital neural network forming a massive human brain floating in space",

"robots working alongside humans in a fully automated futuristic factory",

"satellites orbiting earth transmitting artificial intelligence signals across the planet",

"tiny human standing in front of a gigantic AI neural network visualization",

"dark command center filled with screens showing global AI activity",

"self learning robots assembling more robots in an advanced laboratory",

"massive futuristic data center the size of a city",

"AI controlling global financial markets visualized as glowing data streams",

"scientists watching artificial intelligence predict the future on giant holographic displays",

"futuristic drone swarm coordinated by artificial intelligence",

"superintelligent AI brain floating above earth",

"human silhouette facing a massive glowing artificial intelligence system",

"quantum computer core glowing with energy inside a futuristic lab",

"AI simulation predicting world events on giant holographic displays"
]

# ---------------------------------------------------
# Camera styles (prevents visual repetition)
# ---------------------------------------------------

camera_styles = [

"wide cinematic establishing shot",
"extreme close up shot",
"dramatic low angle shot",
"aerial drone perspective",
"cinematic over the shoulder shot",
"top down perspective",
"epic wide landscape shot",
"macro detail lens shot",
"dynamic cinematic angle",
"tracking cinematic shot"
]

# ---------------------------------------------------
# Inject curiosity spikes every ~7 scenes
# (~28 seconds attention reset)
# ---------------------------------------------------

for i in range(0, len(scenes), 7):

    scenes[i] += " " + random.choice(visual_spikes)


# ---------------------------------------------------
# Cinematic prompt generator
# ---------------------------------------------------

def build_prompt(scene):

    camera = random.choice(camera_styles)

    return f"""
cinematic documentary film still
{camera}

imax cinema camera
anamorphic 35mm lens
captured mid motion
dynamic cinematic composition

ultra photorealistic
extreme detail
8k resolution
high dynamic range

dramatic cinematic lighting
volumetric light rays
deep shadows
high contrast lighting

atmospheric depth
epic scale environment
realistic textures

global technology documentary style
netflix science documentary aesthetic
national geographic cinematic style

subject:
{scene}

mood:
mysterious
dramatic
futuristic
thought provoking

cinematic color grading
professional film production quality
"""


# ---------------------------------------------------
# Generate prompts
# ---------------------------------------------------

prompts = [build_prompt(s) for s in scenes]

prompts

In [ ]:
import requests

try:
    r = requests.get("https://huggingface.co")
    print("Internet working:", r.status_code)
except Exception as e:
    print("Internet error:", e)

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

# -----------------------------------
# Load model
# -----------------------------------

pipe = StableDiffusionPipeline.from_pretrained(
 "runwayml/stable-diffusion-v1-5",
 torch_dtype=torch.float16,
 safety_checker=None,
 cache_dir="/kaggle/working/model_cache"
).to("cuda")

pipe.enable_attention_slicing()

# new syntax (fixes deprecation warning)
pipe.vae.enable_slicing()


# -----------------------------------
# helper: trim prompts to CLIP limit
# -----------------------------------

def trim_prompt(prompt, max_words=60):
    words = prompt.split()
    return " ".join(words[:max_words])


# -----------------------------------
# generate images
# -----------------------------------

images=[]

for i,p in enumerate(prompts):

    p = trim_prompt(p)

    img = pipe(
        p,
        num_inference_steps=35,
        guidance_scale=8.5,
        height=720,
        width=1280,
        negative_prompt="nsfw, nude, porn, blurry, distorted, low quality"
    ).images[0]

    path=f"{BASE}/scenes/scene_{i}.png"
    img.save(path)

    images.append(path)

images

In [ ]:
import cv2
import numpy as np
import os

# safety check
if "images" not in globals():
    raise Exception("Images list not found. Image generation step did not run.")

clips_folder = f"{BASE}/clips"
os.makedirs(clips_folder, exist_ok=True)

clips = []

for i, img_path in enumerate(images):

    img = cv2.imread(img_path)

    h, w, _ = img.shape

    out_path = f"{clips_folder}/clip_{i}.mp4"

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(out_path, fourcc, 30, (w, h))

    for f in range(120):

        scale = 1 + (f * 0.002)

        M = cv2.getRotationMatrix2D((w/2, h/2), 0, scale)

        frame = cv2.warpAffine(img, M, (w, h))

        out.write(frame)

    out.release()

    clips.append(out_path)

clips[:5]

In [ ]:
list_file=f"{BASE}/clips_list.txt"

with open(list_file,"w") as f:

    for c in clips:
        f.write(f"file '{c}'\n")

!ffmpeg -y -f concat -safe 0 \
-i $BASE/clips_list.txt \
-c copy \
$BASE/video/visual.mp4

In [ ]:
!ffmpeg -y -i $BASE/video/visual.mp4 \
-vf "fade=t=in:st=0:d=1,fade=t=out:st=8:d=1" \
$BASE/video/visual_fx.mp4

In [ ]:
!pip install gTTS

In [ ]:
from gtts import gTTS
import os

os.makedirs(f"{BASE}/audio", exist_ok=True)

tts = gTTS(
    text=transcript,
    lang="en",
    slow=False,
    tld="com"
)

tts.save(f"{BASE}/audio/narration.wav")

print("Narration saved:", f"{BASE}/audio/narration.wav")

In [ ]:
!pip install audiocraft
!pip install sentencepiece
!pip install huggingface_hub

In [ ]:
import numpy as np
from scipy.io.wavfile import write
import os

os.makedirs(f"{BASE}/music", exist_ok=True)

sample_rate = 44100
duration = 20

t = np.linspace(0, duration, int(sample_rate * duration))

# simple cinematic pad chords
freqs = [220, 277, 329]  # A minor style chord

signal = sum(np.sin(2*np.pi*f*t) for f in freqs)

# normalize
signal = signal / np.max(np.abs(signal))

write(f"{BASE}/music/music.wav", sample_rate, signal.astype(np.float32))

print("Music generated:", f"{BASE}/music/music.wav")

In [ ]:
!ffmpeg -y \
-i $BASE/video/visual_fx.mp4 \
-i $BASE/audio/narration.wav \
-i $BASE/music/music.wav \
-filter_complex "[1:a][2:a]amix=inputs=2:duration=longest" \
-shortest \
$BASE/video/final_video.mp4

In [ ]:
import shutil

SOURCE = f"{BASE}/video/final_video.mp4"

OUTPUT = "/kaggle/working/final_video.mp4"

shutil.copy(
    SOURCE,
    OUTPUT
)

print("FINAL_VIDEO:", OUTPUT)